# 11 — GoalScope Subspace Probe

Verification notebook for the subspace probe approach:
- Learn per-layer projection matrices $W_l \in \mathbb{R}^{r \times d}$ that capture intent in a low-rank subspace
- Learn soft-prompt token embeddings for the destination context
- Training signal: cross-entropy against labeled intent blocks from `well-known-tasks/labels`

Steps here:
1. Load one task's labeled blocks + generation
2. Find pivot token positions (first token of each block + preceding)
3. Cache source residuals at those positions
4. Verify `SubspaceProbe` module (shapes, ortho loss, patching)
5. Build teacher-forced destination input
6. Single patched forward pass — verify output differs from unpatched
7. Single training step — verify loss + gradient flow
8. Qualitative check — verbalization before vs after a few steps

In [ ]:
import sys
sys.path.append(".")

import json
from pathlib import Path

import torch
import torch.nn.functional as F
from rich import print as rprint
from rich.tree import Tree

from src.models import ModelandTokenizer
from src.tokens import prepare_input, find_token_range
from src.goals.scope_id import IntentBlock
from src.goals.scope_labeler import load_labeled_tree
from src.goals.subspace_probe import (
    SubspaceProbeConfig,
    SubspaceProbe,
    SubspaceCache,
    build_dest_template_parts,
    cache_source_residuals,
    find_block_pivot_positions,
    patched_lm_forward,
    compute_loss,
)
from src.utils.env_utils import DEFAULT_RESULTS_DIR

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
MODEL_KEY    = "Qwen/Qwen2.5-Coder-32B-Instruct"
MODEL_SHORT  = MODEL_KEY.split("/")[-1]
ORACLE_SHORT = "claude-sonnet-4-6"
TEST_TASK    = "atoi"

# Probe hyperparams
R             = 256     # subspace dimension
N_SOFT_TOKENS = 4
PROBE_LAYERS  = list(range(35, 46))   # layers to project (35–45 inclusive)
LAMBDA_ORTHO  = 0.1

# Directories
RESULTS_DIR   = Path(DEFAULT_RESULTS_DIR)
GEN_DIR       = RESULTS_DIR / "well-known-tasks" / "generations" / MODEL_SHORT
LABEL_DIR     = RESULTS_DIR / "well-known-tasks" / "labels" / MODEL_SHORT / ORACLE_SHORT

print(f"gen dir  : {GEN_DIR}")
print(f"label dir: {LABEL_DIR}")

## 1 — Load Model

In [ ]:
mt = ModelandTokenizer(
    model_key=MODEL_KEY,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print(f"n_layer={mt.n_layer}  d_model={mt.n_embd}")
print(f"layer_name_format : {mt.layer_name_format}")
print(f"embedder_name     : {mt.embedder_name}")

## 2 — Load Labeled Data

In [ ]:
# Load generation
with open(GEN_DIR / f"{TEST_TASK}.json") as f:
    gen_data = json.load(f)

# Reconstruct the full code string (same logic as label_well_known_task_blocks.py)
raw_gen        = gen_data["generation"]
code_prefill   = gen_data["prompt"].split("<code>")[-1]
code_generation = code_prefill + raw_gen
if "</code>" in code_generation:
    code_generation = code_generation.split("</code>")[0]

# Full source prompt for the LM (prompt + generation, used for caching residuals)
source_text = gen_data["prompt"] + gen_data["generation"]

print("=== code_generation ===")
print(code_generation)

In [ ]:
# Load labeled block tree
label_path = LABEL_DIR / f"{TEST_TASK}.json"
root_block, full_code, meta = load_labeled_tree(label_path)
print(f"metadata: {meta}")
print(f"full_code (first 200 chars): {full_code[:200]}")

## 3 — Inspect Block Tree

In [ ]:
def print_block_tree(block: IntentBlock, indent: int = 0) -> None:
    span = f"[{block.char_start}:{block.char_end}]"
    label = f"  → '{block.label}'" if block.label else ""
    snippet = full_code[block.char_start:block.char_start+30].replace("\n","↵")
    print("  " * indent + f"{block.kind} {span} `{snippet}...`{label}")
    for child in block.children:
        print_block_tree(child, indent + 1)

print_block_tree(root_block)

## 4 — Find Pivot Token Positions

In [ ]:
# Tokenise the source text and get offset mapping
tokenized_src = prepare_input(
    source_text, return_offsets_mapping=True, tokenizer=mt
)
src_ids        = tokenized_src["input_ids"][0]       # (seq_len,)
offset_mapping = tokenized_src["offset_mapping"][0]  # (seq_len, 2)

print(f"Source token count: {src_ids.shape[0]}")

In [ ]:
# Find pivot positions for all labeled blocks
pivot_entries = find_block_pivot_positions(
    block_tree=root_block,
    source_prompt=source_text,
    code_str=full_code,
    offset_mapping=offset_mapping,
)

print(f"Found {len(pivot_entries)} labeled block(s) with valid pivot positions:\n")
for block, first_tok, prev_tok in pivot_entries:
    first_str = mt.tokenizer.decode([src_ids[first_tok].item()])
    prev_str  = mt.tokenizer.decode([src_ids[prev_tok].item()])
    print(f"  [{block.kind}] label='{block.label}'")
    print(f"    first_tok={first_tok}  token='{first_str}'  |  prev_tok={prev_tok}  token='{prev_str}'")

## 5 — Cache Source Residuals

In [ ]:
# Collect all unique token positions to cache
all_positions = []
for block, first_tok, prev_tok in pivot_entries:
    all_positions.extend([first_tok, prev_tok])
all_positions = sorted(set(all_positions))

print(f"Caching residuals at {len(all_positions)} token positions, {len(PROBE_LAYERS)} layers...")

residual_cache = cache_source_residuals(
    mt=mt,
    source_text=source_text,
    token_positions=all_positions,
    layers=PROBE_LAYERS,
)

# Inspect
sample_pos  = all_positions[0]
sample_layer = PROBE_LAYERS[0]
h = residual_cache[sample_pos][sample_layer]
print(f"\nresidual_cache[pos={sample_pos}][layer={sample_layer}]: shape={h.shape}, dtype={h.dtype}")
print(f"norm = {h.norm().item():.3f}")

In [ ]:
# Build SubspaceCache objects (one per position per block)
subcaches: list[SubspaceCache] = []
for block, first_tok, prev_tok in pivot_entries:
    for tok_pos, is_preceding in [(first_tok, False), (prev_tok, True)]:
        subcaches.append(SubspaceCache(
            task=TEST_TASK,
            block_kind=block.kind,
            label=block.label,
            src_residuals=residual_cache[tok_pos],
            token_pos=tok_pos,
            is_preceding=is_preceding,
        ))

print(f"{len(subcaches)} SubspaceCache entries:")
for sc in subcaches:
    print(f"  [{sc.block_kind}] is_preceding={sc.is_preceding}  label='{sc.label}'")

## 6 — SubspaceProbe Module Verification

In [ ]:
cfg = SubspaceProbeConfig(
    d_model=mt.n_embd,
    r=R,
    layers=PROBE_LAYERS,
    n_soft_tokens=N_SOFT_TOKENS,
    lambda_ortho=LAMBDA_ORTHO,
)
probe = SubspaceProbe(cfg)
print(f"W shape       : {probe.W.shape}")
print(f"soft_tokens   : {probe.soft_tokens.shape}")
total_params = sum(p.numel() for p in probe.parameters())
print(f"Total params  : {total_params:,}")

In [ ]:
# ── Check initial orthogonality ───────────────────────────────────────────────
ortho_loss_init = probe.orthogonality_loss()
print(f"Ortho loss before reorthogonalize : {ortho_loss_init.item():.6f}")

# After explicit reorthogonalize (QR) it should be ~0
probe.reorthogonalize()
ortho_loss_after = probe.orthogonality_loss()
print(f"Ortho loss after  reorthogonalize : {ortho_loss_after.item():.6f}  (should be ~0)")

# Sanity: W @ W^T should be identity
W0 = probe.W[0].detach()
WWT = W0 @ W0.T
eye_err = (WWT - torch.eye(R)).abs().max().item()
print(f"max |W W^T - I| = {eye_err:.2e}  (should be < 1e-5)")

In [ ]:
# ── Check patch_delta formula ─────────────────────────────────────────────────
d = mt.n_embd
h_src_test  = torch.randn(d)
h_dest_test = torch.randn(d)

delta = probe.patch_delta(h_src_test, h_dest_test, layer_idx=0)
patched = probe.patched_residual(h_src_test, h_dest_test, layer_idx=0)

# Verify: delta lies in the subspace (W @ delta^T should round-trip back)
W0 = probe.W[0].detach()
roundtrip_err = (delta - delta @ W0.T @ W0).norm().item()
print(f"delta shape     : {delta.shape}")
print(f"Roundtrip error : {roundtrip_err:.2e}  (should be ~0 when W is orthonormal)")

# Verify: patched = dest + delta
assert torch.allclose(patched, h_dest_test + delta, atol=1e-5)
print("patched = h_dest + delta  ✓")

## 7 — Destination Template

In [ ]:
before_soft_ids, after_soft_ids = build_dest_template_parts(mt)

print(f"before_soft_ids : {before_soft_ids.shape}")
print(f"after_soft_ids  : {after_soft_ids.shape}")
print(f"n_soft_tokens   : {N_SOFT_TOKENS}")
print(f"total prefix len: {before_soft_ids.shape[0] + N_SOFT_TOKENS + after_soft_ids.shape[0]}")
print()
print("before_soft decoded:", repr(mt.tokenizer.decode(before_soft_ids)))
print("after_soft  decoded:", repr(mt.tokenizer.decode(after_soft_ids)))

In [ ]:
# Tokenise the label of the first SubspaceCache entry
sc0 = subcaches[0]
label_ids = torch.tensor(
    mt.tokenizer.encode(sc0.label, add_special_tokens=False),
    dtype=torch.long,
)
print(f"Label text   : '{sc0.label}'")
print(f"Label tok IDs: {label_ids.tolist()}")
print(f"Label decoded: '{mt.tokenizer.decode(label_ids)}'")

## 8 — Patched Forward Pass Verification

Verify that:
1. The patched logits differ from the unpatched baseline
2. The function is differentiable w.r.t. `probe` parameters

In [ ]:
# Move probe to the model's device
device = next(mt._model.parameters()).device
probe = probe.to(device)

# ── Unpatched baseline ────────────────────────────────────────────────────────
# (same setup but with zero source residuals → patch_delta = 0)
d = mt.n_embd
zero_residuals = {l: torch.zeros(d) for l in PROBE_LAYERS}

with torch.no_grad():
    logits_zero = patched_lm_forward(
        mt, probe,
        before_soft_ids, after_soft_ids, label_ids,
        src_residuals=zero_residuals,
    )  # (n_label, vocab)

# ── Real patch (sc0) ──────────────────────────────────────────────────────────
with torch.no_grad():
    logits_patched = patched_lm_forward(
        mt, probe,
        before_soft_ids, after_soft_ids, label_ids,
        src_residuals=sc0.src_residuals,
    )

diff = (logits_patched - logits_zero).abs()
print(f"Logit diff (patched vs zero src):")
print(f"  max  = {diff.max().item():.4f}")
print(f"  mean = {diff.mean().item():.4f}")
print("(Should be non-zero — the patch is active)")

In [ ]:
# ── Gradient flow check ───────────────────────────────────────────────────────
logits_grad = patched_lm_forward(
    mt, probe,
    before_soft_ids, after_soft_ids, label_ids,
    src_residuals=sc0.src_residuals,
)
loss_test, ce_test, ortho_test = compute_loss(logits_grad, label_ids, probe)
loss_test.backward()

print(f"Loss (CE)    = {ce_test.item():.4f}")
print(f"Loss (ortho) = {ortho_test.item():.4f}")
print(f"Loss (total) = {loss_test.item():.4f}")

print(f"\ngrad W norm        : {probe.W.grad.norm().item():.4f}  (non-zero ✓)")
print(f"grad soft_tok norm : {probe.soft_tokens.grad.norm().item():.4f}  (non-zero ✓)")

probe.zero_grad()

## 9 — Single Training Step

In [ ]:
# Re-init probe (fresh start)
probe = SubspaceProbe(cfg).to(device)
probe.reorthogonalize()

optimizer = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=1e-2)

losses = []
for step in range(5):
    optimizer.zero_grad()

    # Use sc0 for this test
    logits = patched_lm_forward(
        mt, probe,
        before_soft_ids, after_soft_ids, label_ids,
        src_residuals=sc0.src_residuals,
    )
    total, ce, ortho = compute_loss(logits, label_ids, probe)

    total.backward()
    torch.nn.utils.clip_grad_norm_(probe.parameters(), 1.0)
    optimizer.step()
    probe.reorthogonalize()  # hard orthogonality constraint

    losses.append(total.item())
    print(f"step {step}  total={total.item():.4f}  ce={ce.item():.4f}  ortho={ortho.item():.4f}")

print(f"\nLoss went {'DOWN ✓' if losses[-1] < losses[0] else 'UP — check config'}")

## 10 — Qualitative Check: Verbalization Before vs After

Generate a short verbalization by greedy decoding from the patched destination prompt.
Even with just a few training steps the output should start shifting toward the label.

In [ ]:
from src.functional import generate_with_patch, PatchSpec
from src.goals.subspace_probe import build_dest_template_parts
import baukit

def verbalize(probe_: SubspaceProbe, sc_: SubspaceCache, max_new_tokens: int = 24) -> str:
    """Greedy decode from the patched destination prompt."""
    dev = next(probe_.parameters()).device
    dtype = next(mt._model.parameters()).dtype

    embed_fn = baukit.get_module(mt._model, mt.embedder_name)
    with torch.no_grad():
        before_emb = embed_fn(before_soft_ids.to(dev)).to(dtype)
        after_emb  = embed_fn(after_soft_ids.to(dev)).to(dtype)
    soft_emb   = probe_.soft_tokens.to(dtype)
    prefix_emb = torch.cat([before_emb, soft_emb, after_emb], dim=0).unsqueeze(0)  # (1, P, d)
    prefix_len = prefix_emb.shape[1]
    patch_start = prefix_len - 1

    # Build layer hooks
    layer_to_idx = {l: i for i, l in enumerate(probe_.config.layers)}
    handles = []
    step_count = [0]

    for l in probe_.config.layers:
        li = layer_to_idx[l]
        module = baukit.get_module(mt._model, mt.layer_name_format.format(l))
        src_h  = sc_.src_residuals[l].to(dev).to(dtype)

        def _hook(mod, inp, output, li_=li, src_h_=src_h):
            hidden = output[0] if isinstance(output, tuple) else output
            rest   = output[1:] if isinstance(output, tuple) else ()
            cur_patch_start = patch_start if hidden.shape[1] >= prefix_len else 0
            region  = hidden[:, cur_patch_start:, :]
            src_exp = src_h_.view(1, 1, -1).expand_as(region)
            patched = region + (src_exp - region) @ probe_.W[li_].T @ probe_.W[li_]
            new_h   = torch.cat([hidden[:, :cur_patch_start, :], patched], dim=1)
            return (new_h,) + rest if rest else new_h

        handles.append(module.register_forward_hook(_hook))

    gen_ids = []
    try:
        with torch.no_grad():
            out = mt._model(inputs_embeds=prefix_emb)
            next_tok = out.logits[0, -1, :].argmax().item()
            gen_ids.append(next_tok)
            # Continue autoregressively
            inp_ids = torch.tensor([[next_tok]], device=dev)
            for _ in range(max_new_tokens - 1):
                out = mt._model(input_ids=inp_ids)
                next_tok = out.logits[0, -1, :].argmax().item()
                eos = mt.tokenizer.eos_token_id
                if next_tok == eos:
                    break
                gen_ids.append(next_tok)
                inp_ids = torch.tensor([[next_tok]], device=dev)
    finally:
        for h in handles:
            h.remove()

    return mt.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

In [ ]:
# Fresh probe (untrained baseline)
probe_untrained = SubspaceProbe(cfg).to(device)
probe_untrained.reorthogonalize()

print(f"Ground truth label : '{sc0.label}'")
print(f"Verbalization (untrained probe)  : '{verbalize(probe_untrained, sc0)}'")
print(f"Verbalization (5-step probe)     : '{verbalize(probe, sc0)}'")

## 11 — Multi-Task Mini Training Loop (Optional)

Load all labeled tasks for this model, build caches, and run a short training loop to verify the pipeline end-to-end before launching the full script.

In [ ]:
import random
from tqdm.auto import tqdm

# Load a small subset of tasks to keep this fast
MINI_TASKS = ["atoi", "binary_search", "fibonacci"]  # adjust as needed

all_caches: list[SubspaceCache] = []

for task in MINI_TASKS:
    gen_path   = GEN_DIR   / f"{task}.json"
    label_path = LABEL_DIR / f"{task}.json"
    if not gen_path.exists() or not label_path.exists():
        print(f"Skipping {task} (missing file)")
        continue

    with open(gen_path) as f:
        gdata = json.load(f)

    raw_gen_t      = gdata["generation"]
    prefill_t      = gdata["prompt"].split("<code>")[-1]
    code_gen_t     = prefill_t + raw_gen_t
    if "</code>" in code_gen_t:
        code_gen_t = code_gen_t.split("</code>")[0]
    src_text_t     = gdata["prompt"] + gdata["generation"]

    root_t, full_code_t, _ = load_labeled_tree(label_path)

    tok_t     = prepare_input(src_text_t, return_offsets_mapping=True, tokenizer=mt)
    src_ids_t = tok_t["input_ids"][0]
    offmap_t  = tok_t["offset_mapping"][0]

    pivots_t  = find_block_pivot_positions(root_t, src_text_t, full_code_t, offmap_t)
    positions_t = sorted({p for _, ft, pt in pivots_t for p in [ft, pt]})

    if not positions_t:
        print(f"No valid pivots for {task}")
        continue

    res_t = cache_source_residuals(mt, src_text_t, positions_t, PROBE_LAYERS)

    for block_t, first_t, prev_t in pivots_t:
        for tok_pos_t, is_prec_t in [(first_t, False), (prev_t, True)]:
            all_caches.append(SubspaceCache(
                task=task,
                block_kind=block_t.kind,
                label=block_t.label,
                src_residuals=res_t[tok_pos_t],
                token_pos=tok_pos_t,
                is_preceding=is_prec_t,
            ))

print(f"\nTotal caches: {len(all_caches)}")
for sc in all_caches:
    print(f"  {sc.task}  [{sc.block_kind}]  pre={sc.is_preceding}  '{sc.label}'")

In [ ]:
# Mini training loop
N_MINI_EPOCHS = 5
LR            = 3e-4

probe_mini = SubspaceProbe(cfg).to(device)
probe_mini.reorthogonalize()
opt_mini   = torch.optim.AdamW(probe_mini.parameters(), lr=LR, weight_decay=1e-2)

for epoch in range(N_MINI_EPOCHS):
    random.shuffle(all_caches)
    epoch_ce = epoch_total = 0.0
    for sc in all_caches:
        lids = torch.tensor(
            mt.tokenizer.encode(sc.label, add_special_tokens=False), dtype=torch.long
        )
        opt_mini.zero_grad()
        logits = patched_lm_forward(
            mt, probe_mini, before_soft_ids, after_soft_ids, lids,
            src_residuals=sc.src_residuals,
        )
        total, ce, _ = compute_loss(logits, lids, probe_mini)
        total.backward()
        torch.nn.utils.clip_grad_norm_(probe_mini.parameters(), 1.0)
        opt_mini.step()
        probe_mini.reorthogonalize()
        epoch_ce    += ce.item()
        epoch_total += total.item()

    n = len(all_caches)
    print(f"Epoch {epoch}  total={epoch_total/n:.4f}  ce={epoch_ce/n:.4f}")

In [ ]:
# Qualitative check across mini tasks
for sc in all_caches[:4]:
    verbalization = verbalize(probe_mini, sc)
    print(f"[{sc.task}|{sc.block_kind}|pre={sc.is_preceding}]")
    print(f"  target : '{sc.label}'")
    print(f"  output : '{verbalization}'")
    print()